# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append(str(Path("..").resolve()))
from src.utils import DATA_PROC, REPORTS, ROOT_DIR, get_logger
from src.model import run_training, compute_metrics

logger = get_logger("modeling_notebook")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

# Load engineered features

In [ ]:
train = pd.read_parquet(DATA_PROC / "train_features.parquet")

y = train["TARGET"].copy()
X = train.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore")

logger.info(f"X: {X.shape} | y distribution: {y.value_counts(normalize=True).round(3).to_dict()}")

from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.2, random_state=42, n_jobs=-1)
X_resampled, y_resampled = smote.fit_resample(X, y)
print(f"After SMOTE — positive rate: {y_resampled.mean():.3f} | rows: {len(y_resampled):,}")

# Run the full training pipeline

In [ ]:
import mlflow
from pathlib import Path
from src.utils import ROOT_DIR

# Confirm tracking URI resolves correctly
db_path = ROOT_DIR / "mlflow" / "mlflow.db"
tracking_uri = f"sqlite:///{db_path.as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
print(f"Tracking URI : {tracking_uri}")
print(f"DB exists    : {db_path.exists()}")

# Confirm experiment creation works
exp = mlflow.set_experiment("credit-risk-intelligence")
print(f"Experiment ID: {exp.experiment_id}")
print(f"Experiment   : {exp.name}")

# Fire a test run with one metric to confirm end-to-end logging
with mlflow.start_run(run_name="connectivity_test") as run:
    mlflow.log_param("test_param", "ok")
    mlflow.log_metric("test_metric", 1.0)
    print(f"Run ID       : {run.info.run_id}")
    print(f"Run status   : {run.info.status}")

print("\n✅ MLflow logging confirmed — ready for full training run")

In [ ]:
fold_models, meta_model, oof_preds, oof_metrics = run_training(X_resampled, y_resampled)

print("\n=== Final OOF Metrics ===")
for k, v in oof_metrics.items():
    print(f"  {k:25s}: {v}")

# Precision-Recall curve

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

precision, recall, _ = precision_recall_curve(y_resampled, oof_preds)
fpr, tpr, _          = roc_curve(y_resampled, oof_preds)
baseline_precision   = y_resampled.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR curve
axes[0].plot(recall, precision, color="#4C9BE8", lw=2, label=f"Model (AUC={oof_metrics['pr_auc']})")
axes[0].axhline(baseline_precision, color="#E8593C", linestyle="--", lw=1.5,
                label=f"Random baseline ({baseline_precision:.3f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend()

# ROC curve
axes[1].plot(fpr, tpr, color="#4C9BE8", lw=2, label=f"Model (AUC={oof_metrics['roc_auc']})")
axes[1].plot([0,1], [0,1], color="gray", linestyle="--", lw=1, label="Random baseline")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()

plt.suptitle("OOF Model Performance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(REPORTS / "06_model_performance_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Top 25 feature importances

In [ ]:
fi_df = pd.read_csv(DATA_PROC / "feature_importance.csv").head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#E8593C" if "EXT_SOURCE" in f or "MISSING" in f or any(
          r in f for r in ["RATIO","TERM","STABILITY","DISAGREEMENT","MEAN","STD"])
          else "#4C9BE8" for f in fi_df["feature"]]

ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel("Feature Importance (gain)")
ax.set_title("Top 25 Feature Importances\n(Red = engineered features)", fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#E8593C", label="Engineered features"),
    Patch(facecolor="#4C9BE8", label="Raw features")
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.savefig(REPORTS / "07_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# Probability calibration check

In [ ]:
from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_resampled, oof_preds, n_bins=15, strategy="quantile"
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(mean_predicted_value, fraction_of_positives,
        "s-", color="#4C9BE8", lw=2, label="LightGBM OOF")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Plot — Is the model well-calibrated?", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / "08_calibration_plot.png", dpi=150, bbox_inches="tight")
plt.show()

# View MLflow run in browser

In [ ]:
# Launch MLflow UI to inspect your tracked experiment
import subprocess
print("Run this in a separate PowerShell terminal:")
print(f"  cd {ROOT_DIR}")
print(f"  mlflow ui --backend-store-uri sqlite:///mlflow/mlflow.db --port 5000")
print(f"\nThen open: http://localhost:5000")